# Data Cleaning and Quality Analysis

**OIBSIP Data Analytics Internship — Task 3**

**Goal:** Take a messy retail transaction dataset, identify quality problems, clean the data and compare the dataset before and after cleaning.

## 1. Import libraries

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

## 2. Load the raw file

In [2]:
# The original file had an extra header row, so header=1 is used.
df = pd.read_csv("data/retail_sales_transactions_raw.csv", header=1)

df.columns = [
    "Order_ID","Customer_Name","Gender","Category","Product","Quantity",
    "Unit_Price","Discount","Order_Date","Sales_Channel","Region","Phone",
    "Product_Name_Duplicate"
]

print("Original shape:", df.shape)
df.head()

Original shape: (499, 13)


,Order_ID,Customer_Name,Gender,Category,Product,Quantity,Unit_Price,Discount,Order_Date,Sales_Channel,Region,Phone,Product_Name_Duplicate
0,1001.0,Sam,Female,GROCERIES,Rice,4.0,499.0,0.0,10-26-2024,Retail,NaN,NaN,Rice
1,1002.0,Sam,NaN,electronics,Tablet,NaN,499.0,NaN,01-02-2024,Wholesale,South,9876543210,Tablet
2,1003.0,John,Male,Electronics,Mobile,NaN,NaN,5.0,03-27-2024,Retail,East,9876543210,Mobile
3,1004.0,John,Female,CLOTHING,Shirt,5.0,NaN,10.0,08-30-2024,Retail,West,NaN,Shirt
4,1005.0,Maria,Female,CLOTHING,Jacket,1.0,NaN,NaN,03-13-2024,Wholesale,North,-9876543119,Jacket


## 3. Remove completely empty rows

In [3]:
df = df.dropna(how="all").copy()

print("Shape after removing completely empty rows:", df.shape)

Shape after removing completely empty rows: (499, 13)


## 4. Create a data-quality report

In [4]:
print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nData types:")
print(df.dtypes)

Missing values:
Order_ID                    0
Customer_Name               0
Gender                    165
Category                    0
Product                     0
Quantity                  150
Unit_Price                160
Discount                  183
Order_Date                  0
Sales_Channel             118
Region                     98
Phone                     203
Product_Name_Duplicate      0
dtype: int64

Duplicate rows: 0

Data types:
Order_ID                  float64
Customer_Name              object
Gender                     object
Category                   object
Product                    object
Quantity                  float64
Unit_Price                float64
Discount                  float64
Order_Date                 object
Sales_Channel              object
Region                     object
Phone                      object
Product_Name_Duplicate     object
dtype: object


## 5. Remove duplicated product column

In [5]:
# Product_Name_Duplicate repeats information already available in Product.
df = df.drop(columns=["Product_Name_Duplicate"])

print("Columns after removing redundant column:")
print(df.columns.tolist())

Columns after removing redundant column:
['Order_ID', 'Customer_Name', 'Gender', 'Category', 'Product', 'Quantity', 'Unit_Price', 'Discount', 'Order_Date', 'Sales_Channel', 'Region', 'Phone']


## 6. Standardise text columns

In [6]:
text_columns = ["Gender", "Category", "Product", "Sales_Channel", "Region", "Customer_Name"]

for col in text_columns:
    df[col] = df[col].astype("string").str.strip()

df["Gender"] = df["Gender"].str.title()
df["Category"] = df["Category"].str.title()
df["Product"] = df["Product"].str.title()
df["Sales_Channel"] = df["Sales_Channel"].str.title()
df["Region"] = df["Region"].str.title()

print("Category values after standardisation:")
print(df["Category"].value_counts(dropna=False))

Category values after standardisation:
Category
Groceries      162
Electronics    143
Clothing       130
Furniture       64
Name: count, dtype: Int64


## 7. Correct numeric and date columns

In [7]:
for col in ["Order_ID", "Quantity", "Unit_Price", "Discount"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce")

print(df.dtypes)

Order_ID                float64
Customer_Name    string[python]
Gender           string[python]
Category         string[python]
Product          string[python]
Quantity                float64
Unit_Price              float64
Discount                float64
Order_Date       datetime64[ns]
Sales_Channel    string[python]
Region           string[python]
Phone                    object
dtype: object


## 8. Handle missing values

In [8]:
# Categorical values
df["Gender"] = df["Gender"].fillna("Unknown")
df["Sales_Channel"] = df["Sales_Channel"].fillna("Unknown")
df["Region"] = df["Region"].fillna("Unknown")

# Numeric values: median is used because it is less affected by extreme values.
for col in ["Quantity", "Unit_Price", "Discount"]:
    df[col] = df[col].fillna(df[col].median())

# Phone numbers are treated as text, not as a mathematical number.
df["Phone"] = df["Phone"].astype("string").str.replace(r"\D", "", regex=True)
df["Phone"] = df["Phone"].replace("", pd.NA).fillna("Unknown")

print("Missing values after treatment:")
print(df.isnull().sum())

Missing values after treatment:
Order_ID         0
Customer_Name    0
Gender           0
Category         0
Product          0
Quantity         0
Unit_Price       0
Discount         0
Order_Date       0
Sales_Channel    0
Region           0
Phone            0
dtype: int64


## 9. Remove exact duplicates

In [9]:
duplicates_before = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)

print("Duplicate rows before:", duplicates_before)
print("Duplicate rows after:", df.duplicated().sum())

Duplicate rows before: 0
Duplicate rows after: 0


## 10. Check possible outliers

In [10]:
numerical_columns = ["Quantity", "Unit_Price", "Discount"]

for col in numerical_columns:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outliers = df[(df[col] < lower) | (df[col] > upper)]

    print(f"{col}: {len(outliers)} possible outliers")
    print(f"Allowed range by IQR rule: {lower:.2f} to {upper:.2f}\n")

Quantity: 0 possible outliers
Allowed range by IQR rule: -1.00 to 7.00

Unit_Price: 0 possible outliers
Allowed range by IQR rule: -251.00 to 1749.00

Discount: 0 possible outliers
Allowed range by IQR rule: -2.50 to 17.50



### Outlier decision

Possible outliers are reported rather than automatically deleted. In retail data, a large quantity or high price can be a legitimate transaction. An analyst should confirm unusual values with the business before removing them.

## 11. Before vs after data quality

In [11]:
# Reload raw data only for comparison.
raw_check = pd.read_csv("data/retail_sales_transactions_raw.csv", header=1)

raw_rows = len(raw_check.dropna(how="all"))
raw_missing = int(raw_check.isnull().sum().sum())
raw_duplicates = int(raw_check.duplicated().sum())

clean_missing = int(df.isnull().sum().sum())
clean_duplicates = int(df.duplicated().sum())

comparison = pd.DataFrame({
    "Metric": ["Rows", "Total Missing Cells", "Duplicate Rows"],
    "Before Cleaning": [raw_rows, raw_missing, raw_duplicates],
    "After Cleaning": [len(df), clean_missing, clean_duplicates]
})

comparison

,Metric,Before Cleaning,After Cleaning
0,Rows,499,499
1,Total Missing Cells,1077,0
2,Duplicate Rows,0,0


## 12. Save the cleaned dataset

In [12]:
df.to_csv("retail_sales_transactions_cleaned.csv", index=False)

print("Cleaned dataset saved as retail_sales_transactions_cleaned.csv")
print("Final shape:", df.shape)

Cleaned dataset saved as retail_sales_transactions_cleaned.csv
Final shape: (499, 12)


## 13. Final observations

- The original dataset contained missing values in several important fields.
- Category, gender and other text fields needed standardisation.
- Numeric columns needed conversion and missing-value treatment.
- The duplicate product-name column was removed because it repeated information.
- Phone numbers were treated as text rather than numerical data.
- Potential outliers were identified with the IQR method but not automatically deleted.
- The final dataset is more consistent and easier to use for analysis.

## 14. Conclusion

Data cleaning is an important part of real analytics work. A clean dataset improves the reliability of charts, calculations and business decisions.